In [1]:
import os
import sys

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from model.model import EcoModel
from model.accounts import REAL_ASSETS, FINANCIAL_ASSETS, TRANSACTIONS

## Experiments

In [3]:
params = {
    "country_number": 2,
    "household_number": 10,
    "psi": 10,
    "upsilon": 1.0,
    "omega": 0.1,
    "initial_wage": 1.0,
    "initial_phi": 1.0,
    "initial_tax_rate": 0.4,
    "cy": 0.9,
    "cd": 0.2,
    "delta": 0.03,
    "cT": 0.4,
    "beta": 2.0,
    "lambda_": 0.2,
    "theta": 0.2,
    "gamma": 0.03,
    "nu": 1.5,
    "rho": 0.95,
    "zeta": 0.1,
    "mu1": 20,
    "mu2": 0.1,
    "iota_l": 1.0,
    "chi": 0.003,
    "iota_b": 0.1,
    "initial_discount_rate": 0.0,
    "initial_bond_rate": 0.001,
    "long_run_rate": 0.0075,
    "xi": 0.8,
    "xi_deltap": 2,
    "inflation_target": 0.005,
    "initial_public_transfer": 0.4,
    "dmax": 0.03,
    "tax_min": 0.35,
    "tax_max": 0.45,
    "g_min": 0.4,
    "g_max": 0.6,
    "eta": 0.03,
    "initial_equity": 10.0,
    "sigma": 4,
}

In [4]:
params["steps"] = 200
model = EcoModel(params)
model.run(seed=1)

AttributeError: 'FiscalAuthority' object has no attribute 'get_cash'

In [ ]:
data = model.output.arrange_variables()
data.head()

In [ ]:
data.dtypes

In [ ]:
import math

data[data.t == 200].groupby("obj_type").agg(math.fsum)

In [ ]:
data[(data.t == 200) & (data.obj_type == "Government")]

In [ ]:
data[(data.t == 200) & (data.obj_type == "CentralBank")]

## Matrice des stocks

### Methodes de creation des matrices de stock

Nous utiliserons 2 fonctions:

* `create_stocks_matrix` pour la creation d'une matrice des stocks a une periode t
* `plot_stocks_consistency` pour la visualisation de la coherence des stocks durant la simulation



In [ ]:
def create_stocks_matrix(raw_data, t):
    # noms
    index_names = ["obj_type"]
    stock_names = list(FINANCIAL_ASSETS + REAL_ASSETS)
    sector_names = ["Household", "Firm", "Bank", "Government", "CentralBank"]

    # extraction des donnees
    period_data = raw_data[raw_data.t == t]
    stock_data = period_data[stock_names + index_names].copy()

    # mise en forme des donnees
    matrix = stock_data.groupby(index_names).agg(math.fsum).transpose()
    matrix = matrix.reindex(sector_names, axis=1, fill_value=0)
    matrix.loc["Net Worth", :] = -1 * matrix.sum()
    matrix.loc[:, "Total"] = matrix.agg(math.fsum, axis=1)
    matrix.loc["Total", :] = matrix.agg(math.fsum, axis=0)
    matrix = matrix.round(2)
    matrix.columns.name = ""
    matrix.index.name = ""
    return matrix

In [ ]:
def plot_stocks_consistency(raw_data):
    # dimensions
    col_num = 2
    var_num = len(FINANCIAL_ASSETS)
    row_num = var_num // col_num + var_num % col_num
    figsize = col_num * 5, row_num * 2

    # extraction
    index_names = ["t"]
    stock_names = list(FINANCIAL_ASSETS)
    final_data = raw_data[stock_names + index_names].groupby(index_names).sum().round(2)

    # visualization
    final_data.plot(subplots=True, layout=(row_num, 2), figsize=figsize)
    plt.tight_layout()
    plt.show()

### Presentation des stocks initiaux et finaux

In [ ]:
print("Matrice initial des stocks")
create_stocks_matrix(data, 0)

In [ ]:
print("Matrice final des stocks")
create_stocks_matrix(data, params["steps"])

### Analyse de la coherence des stocks

In [ ]:
plot_stocks_consistency(data)

## Matrice des flux

### Methodes de creation des matrices des flux

Nous utiliserons 3 fonctions:

* `create_flows_matrix` pour la creation d'une matrice des flux a une periode t
* `plot_flows_consistency` pour la visualisation de la coherence horizontale des flux
* `plot_vertical_consistency` pour la visualisation de la coherence verticale des flux



In [ ]:
def create_flows_matrix(raw_data, t):
    # noms
    index_names = ["obj_type"]
    transaction_names = list(TRANSACTIONS)
    change_names = [f"delta_{name}" for name in FINANCIAL_ASSETS]
    sector_names = ["Household", "Firm", "Bank", "Government", "CentralBank"]
    flow_names = transaction_names + change_names

    # extraction des donnees
    period_data = raw_data[raw_data.t == t]
    flows_data = period_data[flow_names + index_names]

    # mise en forme
    matrix = flows_data.groupby(index_names).sum().transpose()
    matrix = matrix.reindex(sector_names, axis=1, fill_value=0)
    matrix.loc[:, "Total"] = matrix.sum(axis=1)
    matrix.loc["Total", :] = matrix.sum(axis=0)
    matrix = matrix.round(2)
    matrix.columns.name = ""
    matrix.index.name = ""
    return matrix

In [ ]:
def plot_flows_consistency(raw_data):
    # extraction
    index_names = ["t"]
    transaction_names = list(TRANSACTIONS)
    change_names = [f"delta_{name}" for name in FINANCIAL_ASSETS]
    flow_names = transaction_names + change_names
    final_data = raw_data[flow_names + index_names].groupby(index_names).sum().round(2)

    # dimensions
    col_num = 3
    var_num = final_data.shape[1]
    row_num = var_num // col_num + var_num % col_num
    figsize = col_num * 5, row_num * 2

    # visualization
    final_data.plot(subplots=True, layout=(row_num, col_num), figsize=figsize)
    plt.tight_layout()
    plt.show()

### Presentation des flux initiaux et finaux

In [ ]:
print("Matrice initial des flux")
create_flows_matrix(data, 0)

In [ ]:
print("Matrice final des flux")
create_flows_matrix(data, params["steps"])

### Analyse de la coherence des flux

In [ ]:
plot_flows_consistency(data)